
# NLP projekt - nastawienie autorów artykułów

In [ ]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import GridSearchCV
import scipy.stats as stats
from sklearn.model_selection import RandomizedSearchCV
from matplotlib import pyplot as plt
from Helpers.data_set_cleaner import import_amazon_reviews_csv_file, amazon_reviews_split
from Helpers.text_processor import TextProcessor
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Początkowy trening

## Załadowanie danych do treningu

In [2]:
path = r".\..\Data\train.csv"

df = import_amazon_reviews_csv_file(path, 5000)

df

,rating,full_text
0,Neutral,more like funchuck Gave this to my dad for a g...
1,Positive,Inspiring I hope a lot of people hear this cd....
2,Positive,The best soundtrack ever to anything. I'm read...
3,Positive,Chrono Cross OST The music of Yasunori Misuda ...
4,Positive,Too good to be true Probably the greatest soun...
...,...,...
4995,Negative,Waste oF $MONEY$ Waste Of Time & Money... I fo...
4996,Positive,trying to win better This book cuts down the o...
4997,Negative,"don""t waste your money just buy the the lotter..."
4998,Negative,The odds are against you not for you. Winning ...


## Trening

In [2]:
print("[POCZATEK TWORZENIA MODELU]")
data_path = r".\..\Data\train.csv"

data_quantity = 2_000

df = import_amazon_reviews_csv_file(data_path, data_quantity)

X_raw, y = amazon_reviews_split(df)
tp = TextProcessor()
print("[PREPROCESSING (czasochlonne)]")
X_processed = tp.preprocess_series(X_raw)
print("[SPLIT DATA]")
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

# wynik z search grida dla LogisticRegression:
# {'clf__C': 1.0, 'tfidf__max_df': 0.8, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 3)}
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 3), max_df=0.8, min_df=3)),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, C = 1.0))
])
print("[Trenowanie model]")
model_pipeline.fit(X_train, y_train)

print("[KONIEC TWORZENIA MODELU]")

[POCZATEK TWORZENIA MODELU]
[PREPROCESSING (czasochlonne)]
[SPLIT DATA]
[Trenowanie model]
[KONIEC TWORZENIA MODELU]
              precision    recall  f1-score   support

    Negative       0.71      0.80      0.75       152
     Neutral       0.46      0.41      0.44        82
    Positive       0.80      0.75      0.78       166

    accuracy                           0.70       400
   macro avg       0.66      0.65      0.65       400
weighted avg       0.70      0.70      0.70       400



## Ocena uzyskanego modelu

In [5]:
predictions = model_pipeline.predict(X_test)
print(classification_report(y_test, predictions))

#chyba powinno się jakoś sprawdzać czy negatywnych nie bierze jako pozytywnych, no bo jednak jak jest coś negatywne/pozytywne a klasyfikuje się jako neutralne (lub na odwrót) to nie ma tragedii
etykiety = ['Negative', 'Neutral', 'Positive']
cm = confusion_matrix(y_test, predictions, labels=etykiety)

# 3. Rysujemy wykres
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=etykiety)
disp.plot(cmap='Blues', ax=ax, values_format='d')

plt.title("Macierz Pomyłek - Analiza Błędów")
plt.show()

# Testy różnych klasyfikatorów

## LinearSVC

In [15]:
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LinearSVC(class_weight='balanced',random_state=42))
])

param_grid = {
    'tfidf__ngram_range': [(1, 2), (1, 3), (1, 4)],
    'tfidf__max_df': [0.85, 0.9, 0.95],
    'tfidf__min_df': [2, 3, 5],
    'clf__C': [0.1, 1.0, 10]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

y_pred = grid_search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred))

{'clf__C': 1.0, 'tfidf__max_df': 0.95, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 4)}
0.6051662812678007
              precision    recall  f1-score   support

    Negative       0.76      0.84      0.80       421
     Neutral       0.52      0.38      0.44       201
    Positive       0.76      0.79      0.77       378

    accuracy                           0.73      1000
   macro avg       0.68      0.67      0.67      1000
weighted avg       0.71      0.73      0.72      1000



## LogisticRegression

In [8]:
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

param_grid = {
    'tfidf__ngram_range': [(1, 2), (1, 3), (1,4)],
    'tfidf__max_df': [0.85, 0.8],
    'tfidf__min_df': [3, 5],
    'clf__C': [0.1, 1.0]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=3,
    scoring='precision_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

y_pred = grid_search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred))

{'clf__C': 1.0, 'tfidf__max_df': 0.8, 'tfidf__min_df': 3, 'tfidf__ngram_range': (1, 3)}
0.6256679783718568
              precision    recall  f1-score   support

    Negative       0.79      0.78      0.78       421
     Neutral       0.50      0.52      0.51       201
    Positive       0.77      0.76      0.77       378

    accuracy                           0.72      1000
   macro avg       0.69      0.69      0.69      1000
weighted avg       0.72      0.72      0.72      1000



## RandomForestClassifier

In [17]:
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

param_grid = {
    'tfidf__ngram_range': [(1, 2), (1, 3)],
    'tfidf__max_df': [0.85, 0.95],
    'tfidf__min_df': [3, 5],
    'clf__n_estimators': [50, 100],
    'clf__max_depth': [None, 30]
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

y_pred = grid_search.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred))

{'clf__max_depth': 30, 'clf__n_estimators': 100, 'tfidf__max_df': 0.95, 'tfidf__min_df': 5, 'tfidf__ngram_range': (1, 3)}
0.5401999346171009
              precision    recall  f1-score   support

    Negative       0.67      0.79      0.73       421
     Neutral       0.44      0.13      0.20       201
    Positive       0.67      0.78      0.72       378

    accuracy                           0.66      1000
   macro avg       0.59      0.57      0.55      1000
weighted avg       0.62      0.66      0.62      1000



# Trening na większych danych

In [ ]:
from train_model import train_model_full

train_model_full()

# Testy projektu